### 输出阶段的容错处理
---
1. 错误重试 - RetryOutputParser
2. 错误修复 - OutputFixingParser

#### 1. 错误重试 - RetryOutputParser
---

In [ ]:
from langchain_core.exceptions import OutputParserException
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
import os
from dotenv import load_dotenv

# RetryOutputParser 新版本可能不在 langchain_core 中
try:
    from langchain.output_parsers import RetryOutputParser
except ImportError:
    from langchain_classic.output_parsers import RetryOutputParser

load_dotenv()

# 1. 初始化模型
model = ChatOpenAI(
    model="Qwen/Qwen3-8B",  # 请确认这是你服务商实际支持的模型名
    api_key=os.getenv("api_key"),
    base_url=os.getenv("base_url"),
    temperature=0,
)

# 2. 定义结构化输出模型
class Action(BaseModel):
    action: str = Field(description="The action to be taken")
    action_input: str = Field(description="The input for the action")

# 3. 普通 Pydantic 解析器
parser = PydanticOutputParser(pydantic_object=Action)

# 4. Retry 解析器：解析失败时，让 LLM 重新生成符合格式的结果
retry_parser = RetryOutputParser.from_llm(
    parser=parser,
    llm=model,
)

# 5. 提示词模板
prompt = PromptTemplate(
    template=(
        "Answer the user query.\n"
        "{format_instructions}\n"
        "Question: {query}\n"
        "Response:"
    ),
    input_variables=["query"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    },
)

# 6. 构造 PromptValue
prompt_value = prompt.format_prompt(query="北京今天的天气如何?")

# 7. 假设第一次模型输出了错误格式
bad_response = "The weather in Beijing is sunny with a high of 25°C."

# 8. 先演示普通解析失败
try:
    result = parser.parse(bad_response)
except OutputParserException as e:
    print("普通 parser.parse() 解析失败，这是预期行为：")
    print(e)

print("\n" + "=" * 50 + "\n")

# 9. 使用 RetryOutputParser 自动重试
try:
    result = retry_parser.parse_with_prompt(
        completion=bad_response,
        prompt_value=prompt_value,
    )

    print("✅ RetryOutputParser 解析成功：")
    print(result)
    print("action:", result.action)
    print("action_input:", result.action_input)

except Exception as e:
    print("❌ 重试后仍然失败：")
    print(e)

普通 parser.parse() 解析失败，这是预期行为：
Invalid json output: The weather in Beijing is sunny with a high of 25°C.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 


✅ RetryOutputParser 解析成功：
action='provide_weather_info' action_input='The weather in Beijing is sunny with a high of 25°C.'
action: provide_weather_info
action_input: The weather in Beijing is sunny with a high of 25°C.


#### 2. 错误修复 - OutputFixingParser
---


In [11]:
from langchain_classic.output_parsers import OutputFixingParser
from typing import List
from langchain_core.exceptions import OutputParserException
from langchain_core.output_parsers import PydanticOutputParser
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

import os
from dotenv import load_dotenv

load_dotenv()

# 1. 初始化模型
model = ChatOpenAI(
    model="Qwen/Qwen3-8B",  # 请确认这是你服务商实际支持的模型名
    api_key=os.getenv("api_key"),
    base_url=os.getenv("base_url"),
    temperature=0,
)

# 2. 定义结构化输出模型
class Actor(BaseModel):
    name: str = Field(description="name of the actor")
    film_names: List[str] = Field(description="List of films the actor has appeared in")

actor_query = "Generate the filmgraphy for a random actor."
parser = PydanticOutputParser(pydantic_object=Actor)

# 假设生成的结果是错误的格式
bad_response = "The actor is Tom Hanks and he has appeared in films such as Forrest Gump, Cast Away, and Saving Private Ryan."

# 运行的时候抛出错误
# try:
#     result = parser.parse(bad_response) 
# except OutputParserException as e:
#     print("普通 parser.parse() 解析失败，这是预期行为：")
#     print(e)

# 使用OutputFixingParser来修复错误的输出
new_parser = OutputFixingParser.from_llm(
    parser=parser,
    llm=model,
)

# 传入报错信息
new_parser.parse(bad_response)


Actor(name='Tom Hanks', film_names=['Forrest Gump', 'Cast Away', 'Saving Private Ryan'])

In [14]:
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from typing import List
import os
from dotenv import load_dotenv

load_dotenv()

# 1. 初始化模型
model = ChatOpenAI(
    model="Qwen/Qwen3-8B",
    api_key=os.getenv("api_key"),
    base_url=os.getenv("base_url"),
    temperature=0,
)

# 2. 定义 Pydantic 数据模型
class Actor(BaseModel):
    name: str = Field(description="name of the actor")
    film_names: List[str] = Field(description="List of films the actor has appeared in")

# 3. ✅ 直接让模型绑定结构化输出 (无需任何 Parser！)
structured_model = model.with_structured_output(Actor)

# 4. 直接调用，返回的直接就是 Actor 实例
actor_query = "Generate the filmography for a random actor."
result = structured_model.invoke(actor_query)

print(f"✅ Actor Name: {result.name}")
print(f"✅ Films: {result.film_names}")

LengthFinishReasonError: Could not parse response content as the length limit was reached - CompletionUsage(completion_tokens=9070, prompt_tokens=17, total_tokens=9087, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=878, rejected_prediction_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cache_write_tokens=None, cached_tokens=0), prompt_cache_hit_tokens=0, prompt_cache_miss_tokens=17)